# Сбор дополнительных данных и разметка

После EDA мы собрали дополнительные сообщения и кадры фотоловушек. Каждая из нас
разметила одинаковые задания в Label Studio. Потом мы сравнили ответы, просмотрели
спорные примеры и выбрали итоговые метки и рамки.

Здесь собран порядок работы по сохранённым файлам и решениям. Команды запуска
Label Studio восстановлены по истории терминала. Ячейки заново читают
экспорты, считают согласованность и проверяют итоговый набор.


In [1]:
from collections import Counter
import csv
import hashlib
import json
import math
from pathlib import Path
from statistics import mean, median
import unicodedata

cwd = Path.cwd().resolve()
ROOT = next(
    path for path in (cwd, *cwd.parents)
    if (path / "data/annotated_data").is_dir() and (path / "notebooks/02_collection_and_annotation.ipynb").is_file()
)

def read_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def read_csv(relative_path):
    with (ROOT / relative_path).open(encoding="utf-8-sig", newline="") as source:
        return list(csv.DictReader(source))

def read_jsonl(relative_path):
    return [
        json.loads(line)
        for line in (ROOT / relative_path).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

def table(headers, rows):
    rows = [[str(value) for value in row] for row in rows]
    widths = [
        max(len(str(header)), *(len(row[index]) for row in rows))
        for index, header in enumerate(headers)
    ]
    print(" | ".join(str(value).ljust(width) for value, width in zip(headers, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))

print("Файлы найдены. Для этих ячеек хватает стандартной библиотеки Python.")


Файлы найдены. Для этих ячеек хватает стандартной библиотеки Python.


## Сообщения

Из Telegram отобрали 70 сообщений из учебных и общих переписок. Ещё 53 сообщения
взяли из личных SMS. При отборе смотрели на содержание и убирали персональные
данные. Для каждого сообщения сохранили источник и отдельный ID.

Подборку дополнили 77 сообщениями SMS Spam Collection. Корпус выбрали через Kaggle,
а исходный файл скачали из UCI. Эти сообщения перевели с английского на русский
с помощью ИИ. Ассистент сверил переводы с оригиналами. В таблице остались номер
строки UCI, обезличенный оригинал и способ перевода.

Всего подготовили 204 записи. Три неоднозначных сообщения и один близкий повтор
отложили, поэтому на разметку отправили 200. По предварительным меткам получилось
по 100 примеров каждого класса. Окончательные ответы ставили при разметке.

В data/real/telegram/messages_200_with_sources.csv сохранили тексты и происхождение.
В annotation_id записан ID задания вида TG071, а sample_id хранит ID записи
при сборе. Эти поля связаны, но имеют разные значения.

Исходный корпус и сведения о нём доступны на [странице UCI](https://archive.ics.uci.edu/dataset/228/sms+spam+collection).


In [2]:
text_sources = read_csv("data/real/telegram/messages_200_with_sources.csv")
collection = read_json("data/real/telegram/collection_summary.json")
deferred = read_csv("data/real/telegram/deferred_4_with_sources.csv")

assert len(text_sources) == 200
assert len({row["annotation_id"] for row in text_sources}) == 200
assert len(deferred) == 4

table(["Источник", "Число сообщений"], [
    ["Telegram", collection["main_sources"]["telegram_not_spam"]],
    ["Личные SMS", collection["main_sources"]["user_sms_spam"]],
    ["UCI, перевод обычных сообщений", collection["main_sources"]["translated_uci_not_spam"]],
    ["UCI, перевод спама", collection["main_sources"]["translated_uci_spam"]],
    ["Отложены", len(deferred)],
])


Источник                       | Число сообщений
-------------------------------+----------------
Telegram                       | 70             
Личные SMS                     | 53             
UCI, перевод обычных сообщений | 30             
UCI, перевод спама             | 47             
Отложены                       | 4              


При подготовке проверили точные повторы после нормализации Unicode, регистра
и пробелов. В сохранённой проверке от 15 ноября 2026 года среди 204 записей
не нашлось точных совпадений с 21 318 сообщениями базового датасета.
Поиск смысловых повторов относительно всей базы тогда не проводили.

Для переводов сохранили лицензию UCI CC BY 4.0. В таблице Telegram
и личных SMS общая лицензия не установлена. Даты получения личных SMS неизвестны,
а 15 ноября указано как дата сбора подборки.


In [3]:
def normalize(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())

normalized = [normalize(row["text"]) for row in text_sources]
assert len(set(normalized)) == len(normalized)
overlap = read_json("data/real/telegram/base_overlap_check.json")
assert overlap["exact_normalized_matches"] == []

print("Точных повторов среди 200 сообщений:", len(normalized) - len(set(normalized)))
print("Сохранённая сверка с базой:", overlap["checked_on"])
print("Проверено записей:", overlap["selection_rows"])
print("Размер базы:", overlap["base_rows"])
print("Точных совпадений с базой:", len(overlap["exact_normalized_matches"]))


Точных повторов среди 200 сообщений: 0
Сохранённая сверка с базой: 2026-11-15
Проверено записей: 204
Размер базы: 21318
Точных совпадений с базой: 0


## Кадры фотоловушек

Для ручной разметки подготовили 140 кадров. Часть взяли из открытых коллекций
Kruger, Wellington и ENA24, часть добавили из присланных изображений.
В data/real/caltech/sources.csv внесли имя файла, источник, исходный ID,
ссылку, дату обращения, лицензию и SHA-256, когда эти сведения были известны.

Для заданий изображения получили нейтральные имена IMG001, IMG002 и далее.
Связь с оригинальными файлами хранится в
label_studio/06_sources/images/image_sources_and_mapping.json.
Там также записаны размеры и контрольные суммы.

По сообщению автора, 28 снимков отобраны вручную в интернете. Их URL,
дата обращения и лицензия не сохранились. В общем журнале это отмечено отдельно.


In [4]:
image_mapping = read_json("label_studio/06_sources/images/image_sources_and_mapping.json")
mapping_by_id = {row["sample_id"]: row for row in image_mapping}
assert len(mapping_by_id) == len(image_mapping) == 140

source_counts = Counter(row["source_record"]["source"] for row in image_mapping)
table(["Источник кадров", "Количество"], sorted(source_counts.items()))
missing = sum(not row["source_record"].get("license") for row in image_mapping)
print("\nКадров без записанной лицензии:", missing)

for row in image_mapping:
    path = ROOT / row["repo_path"]
    assert path.is_file(), row["sample_id"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["sha256"], row["sample_id"]
print("Все 140 файлов читаются и совпадают с сохранёнными SHA-256.")


Источник кадров                  | Количество
---------------------------------+-----------
ENA24                            | 18        
Kruger                           | 50        
Pinterest по данным пользователя | 12        
Wellington                       | 44        
Вложение пользователя            | 15        
Свежая загрузка                  | 1         

Кадров без записанной лицензии: 28
Все 140 файлов читаются и совпадают с сохранёнными SHA-256.


## Запуск Label Studio

Label Studio установили отдельно от окружения EDA, в
~/Documents/label-studio-coursework. В истории терминала сохранились команды
создания окружения Python 3.12, установки версии 1.23.2 и запуска.

    mkdir -p ~/Documents/label-studio-coursework
    cd ~/Documents/label-studio-coursework
    ~/.local/bin/uv venv --python 3.12 .venv
    ~/.local/bin/uv pip install --python .venv/bin/python 'label-studio==1.23.2'
    ./.venv/bin/label-studio start --data-dir ./ls-data --port 8080

После запуска открывали http://localhost:8080. Терминал с сервером оставался
открытым во время разметки. Проекты и ответы сохранялись в ls-data.
При повторном запуске достаточно перейти в ту же папку и выполнить последнюю
команду. Здесь она приведена как запись использованного запуска.

Для сообщений и фото создали отдельные проекты. Через Data Import загрузили
label_studio/01_texts/text_tasks_200.json и
label_studio/02_images/image_tasks_140.json. В Labeling Setup → Code вставили
соответственно text_labeling.xml и image_labeling.xml из этих же папок.

В обычном наборе фото открывались по ссылкам GitHub. Для работы с локальными
файлами также подготовили image_tasks_140_offline.json и serve_images.py.
Офлайн-вариант запускался отдельной командой из папки label_studio/02_images.

    python3 serve_images.py

Этот сервер отдаёт изображения на 127.0.0.1:8765. Его наличие подтверждается
файлами комплекта, но отдельный лог его фактического запуска не сохранён.


In [5]:
text_tasks = read_json("label_studio/01_texts/text_tasks_200.json")
image_tasks = read_json("label_studio/02_images/image_tasks_140.json")
assert len(text_tasks) == 200 and len(image_tasks) == 140

for tasks, fields in [
    (text_tasks, {"sample_id", "text"}),
    (image_tasks, {"sample_id", "image"}),
]:
    assert len({task["data"]["sample_id"] for task in tasks}) == len(tasks)
    assert all(set(task["data"]) == fields for task in tasks)
    assert all(not task.get("annotations") and not task.get("predictions") for task in tasks)

table(["Проект", "Заданий", "Поля задания"], [
    ["Сообщения", len(text_tasks), "sample_id, text"],
    ["Фото", len(image_tasks), "sample_id, image"],
])
print("\nВ заданиях нет предварительных ответов и готовых рамок.")


Проект    | Заданий | Поля задания    
----------+---------+-----------------
Сообщения | 200     | sample_id, text 
Фото      | 140     | sample_id, image

В заданиях нет предварительных ответов и готовых рамок.


## Два прохода разметки

Обе разметчицы получили одинаковые задания. Тексту присваивали spam или
not_spam. На фото ставили отдельную рамку animal на каждую различимую особь,
а кадру выбирали статус animal, empty или needs_review.
Пустой кадр сохраняли без рамок.

После Submit выгрузили полные JSON через Export → JSON. Ответы лежат в
data/annotated_data/telegram/text_A.json, text_B.json и
data/annotated_data/photo/images_A.json, animals_B.json.
Во время работы некоторые спорные фото A обсуждались с ИИ.

У IMG021 после первого экспорта A исправили статус с animal на empty.
Исходный экспорт сохранили отдельно в
data/annotated_data/photo/original_exports/images_A_2026-11-17.json.
Поэтому согласованность до исправления и после него считаем отдельно.


Сначала прочитаем JSON и проверим, что на каждый ID сохранён один ответ. Функция unpack достаёт метку, время, комментарий и рамки из экспорта Label Studio.


In [6]:
def require(condition, message):
    if not condition:
        raise ValueError(str(message))


def load(path):
    tasks = json.loads(path.read_text(encoding='utf-8'))
    require(isinstance(tasks, list), f'{path}: expected a full JSON task list, not JSON_MIN')
    result = {}
    for task in tasks:
        sid = task['data']['sample_id']
        require(sid not in result, ('duplicate_id', sid))
        require(len(task.get('annotations', [])) == 1, ('expected_one_annotation', sid))
        require(not task['annotations'][0].get('was_cancelled'), ('cancelled', sid))
        result[sid] = task
    return result


def unpack(task, kind):
    annotation = task['annotations'][0]
    results = annotation['result']
    field = 'label' if kind == 'text' else 'frame_status'

    def choice(name):
        entries = [r for r in results if r.get('from_name') == name]
        if not entries:
            return None
        require(len(entries) == 1, ('multiple_choice_results', task['data']['sample_id'], name))
        values = entries[0]['value'].get('choices', [])
        require(len(values) == 1, (task['data']['sample_id'], name, values))
        return values[0]

    flags = {r['id']: sorted(r['value'].get('choices', []))
             for r in results if r.get('from_name') == 'box_flags'}
    boxes = []
    for result in results:
        if result.get('type') != 'rectanglelabels':
            continue
        value = result['value']
        box = {key: float(value[key]) for key in ('x', 'y', 'width', 'height')}
        require(all(math.isfinite(v) for v in box.values()), ('nonfinite_box', task['data']['sample_id']))
        require(value.get('rotation', 0) == 0, ('rotated_box', task['data']['sample_id']))
        require(value.get('rectanglelabels') == ['animal'], ('unexpected_box_label', task['data']['sample_id']))
        require(min(box['x'], box['y']) >= -0.001 and box['width'] > 0 and box['height'] > 0,
                ('invalid_box', task['data']['sample_id']))
        require(box['x'] + box['width'] <= 100.001 and box['y'] + box['height'] <= 100.001,
                ('box_outside_image', task['data']['sample_id']))
        require(result.get('original_width', 0) > 0 and result.get('original_height', 0) > 0,
                ('missing_image_dimensions', task['data']['sample_id']))
        box.update({'id': result['id'], 'flags': flags.get(result['id'], []),
                    'original_width': result['original_width'],
                    'original_height': result['original_height']})
        boxes.append(box)
    record = {
        'status': choice(field),
        'confidence': choice('confidence'),
        'comment': '\n'.join(s for r in results if r.get('from_name') == 'comment'
                             for s in r['value'].get('text', [])),
        'view_adjusted': choice('view_adjusted'),
        'lead_time_seconds': annotation.get('lead_time'),
        'boxes': boxes,
    }
    valid = ('not_spam', 'spam', 'needs_review', 'excluded') if kind == 'text' else ('animal', 'empty', 'needs_review', 'excluded')
    require(record['status'] in valid, ('invalid_status', task['data']['sample_id'], record['status']))
    return record


In [7]:
export_paths = {
    "text_A": ROOT / "data/annotated_data/telegram/text_A.json",
    "text_B": ROOT / "data/annotated_data/telegram/text_B.json",
    "images_A_original": ROOT / "data/annotated_data/photo/original_exports/images_A_2026-11-17.json",
    "images_A_current": ROOT / "data/annotated_data/photo/images_A.json",
    "images_B": ROOT / "data/annotated_data/photo/animals_B.json",
}

exports = {name: load(path) for name, path in export_paths.items()}
text_ids = {task["data"]["sample_id"] for task in text_tasks}
image_ids = {task["data"]["sample_id"] for task in image_tasks}
assert set(exports["text_A"]) == set(exports["text_B"]) == text_ids
assert set(exports["images_A_original"]) == set(exports["images_A_current"]) == set(exports["images_B"]) == image_ids

text_pairs = {}
image_pairs = {}
for sid in sorted(text_ids):
    assert exports["text_A"][sid]["data"] == exports["text_B"][sid]["data"]
    text_pairs[sid] = {
        who: unpack(exports[key][sid], "text")
        for who, key in [("A", "text_A"), ("B", "text_B")]
    }
for sid in sorted(image_ids):
    assert exports["images_A_current"][sid]["data"] == exports["images_B"][sid]["data"] == exports["images_A_original"][sid]["data"]
    image_pairs[sid] = {
        who: unpack(exports[key][sid], "images")
        for who, key in [("A", "images_A_current"), ("B", "images_B"), ("original_A", "images_A_original")]
    }
print("Сопоставлены полные экспорты: 200 сообщений и 140 кадров.")


Сопоставлены полные экспорты: 200 сообщений и 140 кадров.


Время взяли из annotations[].lead_time в экспортах Label Studio.
Это длительность по счётчику программы. В неё могли входить паузы.


In [8]:
time_rows = []
for key in ["text_A", "text_B", "images_A_current", "images_B"]:
    times = [task["annotations"][0].get("lead_time") for task in exports[key].values()]
    assert all(isinstance(value, (int, float)) and value > 0 for value in times)
    total = round(sum(times))
    hours, rest = divmod(total, 3600)
    minutes, seconds = divmod(rest, 60)
    time_rows.append([key, len(times), f"{hours:02}:{minutes:02}:{seconds:02}"])
table(["Экспорт", "Заданий", "Сумма lead_time"], time_rows)


Экспорт          | Заданий | Сумма lead_time
-----------------+---------+----------------
text_A           | 200     | 00:35:23       
text_B           | 200     | 00:22:05       
images_A_current | 140     | 02:11:11       
images_B         | 140     | 00:57:04       


## Сравнение ответов

Ответы сопоставляем по data.sample_id. Внутренние номера проектов Label Studio
могут различаться, поэтому по ним ответы не объединяли.

Для меток считаем долю совпадений и κ Коэна по парам с определёнными метками.
Пары с needs_review показываем отдельно. Если ожидаемое согласие равно единице
или подходящих пар нет, κ не определена.

Для рамок используем порог IoU 0,5. Сначала находим максимальное число пар,
затем выбираем сопоставление с наибольшей суммой IoU. Каждая рамка входит
максимум в одну пару. Функция optimal_match решает эту задачу через поток
минимальной стоимости.

В приложенном кодбуке v1.0 описан другой порядок подсчёта с включением uncertain
и жадным сопоставлением рамок. Ниже сохранён способ расчёта, которым пользовались
при сравнении экспортов.


Расчёт совпадений и κ Коэна.


In [9]:
def binary_metrics(rows, labels):
    eligible = [(a, b) for a, b in rows if a in labels and b in labels]
    n = len(eligible)
    matrix = [[sum(a == left and b == right for a, b in eligible) for right in labels] for left in labels]
    agreement = sum(a == b for a, b in eligible) / n if n else None
    expected = sum(sum(a == label for a, _ in eligible) * sum(b == label for _, b in eligible)
                   for label in labels) / n**2 if n else None
    kappa = (agreement - expected) / (1 - expected) if n and not math.isclose(expected, 1) else None
    return {'total_pairs': len(rows), 'comparable_pairs': n,
            'coverage': n / len(rows) if rows else None,
            'matching_pairs': sum(a == b for a, b in eligible),
            'disagreeing_pairs': sum(a != b for a, b in eligible),
            'nonbinary_pairs': len(rows) - n, 'agreement': agreement, 'cohen_kappa': kappa,
            'matrix_labels': labels, 'matrix_rows_A_columns_B': matrix,
            'status_counts_A': dict(Counter(a for a, _ in rows)),
            'status_counts_B': dict(Counter(b for _, b in rows))}


Пересечение рамок и их сопоставление. Координаты берём в процентах от одного кадра, поэтому IoU совпадает с расчётом в пикселях.


In [10]:
def iou(a, b):
    left = max(a['x'], b['x'])
    top = max(a['y'], b['y'])
    right = min(a['x'] + a['width'], b['x'] + b['width'])
    bottom = min(a['y'] + a['height'], b['y'] + b['height'])
    intersection = max(0, right - left) * max(0, bottom - top)
    union = a['width'] * a['height'] + b['width'] * b['height'] - intersection
    return intersection / union


def optimal_match(matrix, b_count, threshold=0.5):
    """Min-cost maximum flow: maximize pair count first, then summed IoU."""
    a_count = len(matrix)
    source, sink = a_count + b_count, a_count + b_count + 1
    graph = [[] for _ in range(sink + 1)]

    def edge(u, v, cost):
        forward = [v, len(graph[v]), 1, cost]
        graph[u].append(forward)
        graph[v].append([u, len(graph[u]) - 1, 0, -cost])
        return forward

    links = []
    for i in range(a_count):
        require(len(matrix[i]) == b_count, 'Invalid IoU matrix dimensions')
        edge(source, i, 0)
        for j in range(b_count):
            if matrix[i][j] >= threshold:
                links.append((i, j, edge(i, a_count + j, -matrix[i][j])))
    for j in range(b_count):
        edge(a_count + j, sink, 0)
    while True:
        distance, parent = [math.inf] * len(graph), [None] * len(graph)
        distance[source] = 0
        for _ in range(len(graph) - 1):
            changed = False
            for u, edges in enumerate(graph):
                if not math.isfinite(distance[u]):
                    continue
                for k, (v, _, capacity, cost) in enumerate(edges):
                    if capacity and distance[u] + cost < distance[v] - 1e-12:
                        distance[v], parent[v] = distance[u] + cost, (u, k)
                        changed = True
            if not changed:
                break
        if parent[sink] is None:
            break
        node = sink
        while node != source:
            u, k = parent[node]
            selected = graph[u][k]
            selected[2] -= 1
            graph[node][selected[1]][2] += 1
            node = u
    pairs = tuple((i, j) for i, j, selected in links if selected[2] == 0)
    return len(pairs), sum(matrix[i][j] for i, j in pairs), pairs


def compare_boxes(a, b):
    matrix = [[iou(left, right) for right in b] for left in a]
    count, total, pairs = optimal_match(matrix, len(b))
    matched = [{'A_index': i, 'B_index': j, 'iou': matrix[i][j],
                'flags_A': a[i]['flags'], 'flags_B': b[j]['flags']} for i, j in pairs]
    return {'boxes_A': len(a), 'boxes_B': len(b), 'matched_count': count,
            'matched': matched,
            'unmatched_A': [i for i in range(len(a)) if i not in {i for i, _ in pairs}],
            'unmatched_B': [j for j in range(len(b)) if j not in {j for _, j in pairs}],
            'matching_f1': 2 * count / (len(a) + len(b)) if a or b else None,
            'mean_matched_iou': total / count if count else None,
            'median_matched_iou': median(item['iou'] for item in matched) if matched else None}


Общие показатели рамок считаем на кадрах, где обе разметчицы выбрали animal или empty. Пустые кадры учитываются в покрытии, а средний IoU считается по найденным парам рамок.


In [11]:
def box_summary(rows):
    comparable = [row for row in rows if row['A']['status'] in ('animal', 'empty') and row['B']['status'] in ('animal', 'empty')]
    a_count = sum(row['box_comparison']['boxes_A'] for row in comparable)
    b_count = sum(row['box_comparison']['boxes_B'] for row in comparable)
    matched = [pair for row in comparable for pair in row['box_comparison']['matched']]
    return {'scope': 'Frames with a binary status from both annotators; needs_review/excluded frames excluded and reported separately.',
            'iou_threshold': 0.5, 'matching': 'Maximum cardinality, then maximum summed IoU; one-to-one.',
            'comparable_frames': len(comparable),
            'frames_with_two_empty_box_sets': sum(not row['A']['boxes'] and not row['B']['boxes'] for row in comparable),
            'frames_with_any_boxes': sum(bool(row['A']['boxes'] or row['B']['boxes']) for row in comparable),
            'boxes_A': a_count, 'boxes_B': b_count, 'matched_pairs': len(matched),
            'unmatched_A': a_count - len(matched), 'unmatched_B': b_count - len(matched),
            'matching_f1': 2 * len(matched) / (a_count + b_count) if a_count + b_count else None,
            'mean_matched_iou': mean(pair['iou'] for pair in matched) if matched else None,
            'median_matched_iou': median(pair['iou'] for pair in matched) if matched else None,
            'pairs_below_review_iou_075': sum(pair['iou'] < 0.75 for pair in matched),
            'matched_pairs_with_different_flags': sum(pair['flags_A'] != pair['flags_B'] for pair in matched)}


In [12]:
text_metrics = binary_metrics(
    [(row["A"]["status"], row["B"]["status"]) for row in text_pairs.values()],
    ["not_spam", "spam"],
)
original_metrics = binary_metrics(
    [(row["original_A"]["status"], row["B"]["status"]) for row in image_pairs.values()],
    ["empty", "animal"],
)
current_metrics = binary_metrics(
    [(row["A"]["status"], row["B"]["status"]) for row in image_pairs.values()],
    ["empty", "animal"],
)
table(["Сравнение", "Совпало", "Определённых пар", "Согласие", "Каппа"], [
    [name, metric["matching_pairs"], metric["comparable_pairs"],
     f'{metric["agreement"]:.2%}', f'{metric["cohen_kappa"]:.4f}']
    for name, metric in [
        ["Тексты", text_metrics],
        ["Фото, исходный A", original_metrics],
        ["Фото, исправленный A", current_metrics],
    ]
])
print("\nКадров с needs_review у B:",
      sum(row["B"]["status"] == "needs_review" for row in image_pairs.values()))

box_rows = []
for sid, row in image_pairs.items():
    box_rows.append({
        "sample_id": sid, "A": row["A"], "B": row["B"],
        "box_comparison": compare_boxes(row["A"]["boxes"], row["B"]["boxes"]),
    })
box_metrics = box_summary(box_rows)
table(["Показатель рамок", "Значение"], [
    ["Рамок A / B на кадрах с определёнными статусами",
     f'{box_metrics["boxes_A"]} / {box_metrics["boxes_B"]}'],
    ["Сопоставлено", box_metrics["matched_pairs"]],
    ["Без пары A / B", f'{box_metrics["unmatched_A"]} / {box_metrics["unmatched_B"]}'],
    ["Средний IoU сопоставленных пар", f'{box_metrics["mean_matched_iou"]:.4f}'],
    ["Медианный IoU", f'{box_metrics["median_matched_iou"]:.4f}'],
])


Сравнение            | Совпало | Определённых пар | Согласие | Каппа 
---------------------+---------+------------------+----------+-------
Тексты               | 199     | 200              | 99.50%   | 0.9900
Фото, исходный A     | 130     | 135              | 96.30%   | 0.9259
Фото, исправленный A | 131     | 135              | 97.04%   | 0.9407

Кадров с needs_review у B: 5
Показатель рамок                                | Значение 
------------------------------------------------+----------
Рамок A / B на кадрах с определёнными статусами | 111 / 110
Сопоставлено                                    | 106      
Без пары A / B                                  | 5 / 4    
Средний IoU сопоставленных пар                  | 0.8642   
Медианный IoU                                   | 0.8860   


## Просмотр расхождений и итоговые решения

Первичная проверка отметила 39 примеров. Из списка убрали 12 кадров, где ответы
совпали и причиной проверки была только низкая уверенность без пояснения.
Ещё четыре кадра отличались только дополнительными признаками
occluded / truncated.

Для разбора оставили одно сообщение и 22 кадра. Смотрели разные статусы,
число рамок, рамки без пары и пары с IoU ниже 0,75. Этот порог помогал найти
различия в границах. После просмотра выбирали вариант A или B для всего кадра.

Кадры показали рядом с рамками обеих разметчиц на странице просмотра.
Принятые решения записали отдельно в
artifacts/annotation_comparison/adjudication_template.json.
В TG071 оставили not_spam, потому что сообщение обсуждает учебный предмет.
IMG064, IMG099 и IMG117 исключили как спорные.


Соберём причины проверки для каждого примера. Для итогового разбора оставим различия в метках, статусах и рамках.


In [13]:
def reasons(a, b, kind, comparison=None):
    result = []
    binary = ('spam', 'not_spam') if kind == 'text' else ('animal', 'empty')
    if a['status'] not in binary or b['status'] not in binary:
        result.append('unresolved_status')
    elif a['status'] != b['status']:
        result.append('label_disagreement')
    for name, annotation in [('A', a), ('B', b)]:
        if annotation['confidence'] == 'low':
            result.append('low_confidence_' + name)
        if (annotation['confidence'] == 'low' or annotation['status'] not in binary) and not annotation['comment'].strip():
            result.append('missing_comment_' + name)
        if kind == 'images' and annotation['status'] == 'animal' and not annotation['boxes']:
            result.append('animal_without_bbox_' + name)
        if kind == 'images' and annotation['status'] == 'empty' and annotation['boxes']:
            result.append('empty_with_bbox_' + name)
    if comparison:
        if comparison['unmatched_A'] or comparison['unmatched_B']:
            result.append('unmatched_boxes')
        if any(pair['iou'] < 0.75 for pair in comparison['matched']):
            result.append('matched_iou_below_075')
        if any(pair['flags_A'] != pair['flags_B'] for pair in comparison['matched']):
            result.append('different_box_flags')
    return result


In [14]:
decisions = read_json("artifacts/annotation_comparison/adjudication_template.json")
decision_by_id = {row["sample_id"]: row for row in decisions["items"]}
assert len(decision_by_id) == len(decisions["items"]) == 23

review_reasons = {
    "label_disagreement", "unresolved_status", "unmatched_boxes", "matched_iou_below_075"
}
queue_ids = set()
for sid, row in text_pairs.items():
    if review_reasons.intersection(reasons(row["A"], row["B"], "text")):
        queue_ids.add(sid)
for row in box_rows:
    sid = row["sample_id"]
    if review_reasons.intersection(
        reasons(row["A"], row["B"], "images", row["box_comparison"])
    ):
        queue_ids.add(sid)
assert queue_ids == set(decision_by_id)

decision_table = []
for item in decisions["items"]:
    sid = item["sample_id"]
    if item["final_status"] == "excluded":
        choice = "исключён"
    elif item["dataset"] == "text":
        choice = item["final_label"]
    else:
        matches = [
            who for who in ("A", "B")
            if image_pairs[sid][who]["status"] == item["final_label"]
            and image_pairs[sid][who]["boxes"] == item["final_boxes"]
        ]
        assert len(matches) == 1, sid
        choice = matches[0]
    boxes = item["final_boxes"]
    decision_table.append([
        sid, choice, item["final_label"] or "—",
        len(boxes) if boxes is not None else "—",
    ])
table(["ID", "Выбрали", "Итоговая метка", "Рамок"], decision_table)


ID     | Выбрали  | Итоговая метка | Рамок
-------+----------+----------------+------
TG071  | not_spam | not_spam       | —    
IMG027 | A        | animal         | 5    
IMG029 | B        | animal         | 2    
IMG031 | A        | empty          | 0    
IMG033 | B        | animal         | 4    
IMG043 | A        | empty          | 0    
IMG054 | B        | animal         | 4    
IMG064 | исключён | —              | —    
IMG079 | B        | animal         | 5    
IMG086 | A        | animal         | 2    
IMG099 | исключён | —              | —    
IMG116 | A        | empty          | 0    
IMG117 | исключён | —              | —    
IMG123 | A        | animal         | 1    
IMG129 | A        | animal         | 1    
IMG016 | A        | animal         | 3    
IMG035 | A        | animal         | 1    
IMG052 | B        | animal         | 1    
IMG069 | A        | animal         | 5    
IMG072 | A        | animal         | 7    
IMG084 | B        | animal         | 2    
IMG088 | B 

## Сборка итогового набора

Для разобранных примеров взяли сохранённые решения. Для остальных сообщений
оставили совпадающую метку. У остальных кадров статусы и число рамок совпали,
и в итоговый набор взяли рамки A.

Сообщения записали построчно в data/real/final/texts.jsonl. В каждой строке
есть ID, текст, итоговая метка и сведения об источнике.
Для фото скопировали исходные файлы и подготовили COCO-разметку.
Координаты Label Studio перевели из процентов в пиксели.
Например, x_px = x_percent × width / 100.
Пустые кадры сохранили с нулём рамок.

Следующие ячейки заново собирают ожидаемые метки и рамки в памяти и сравнивают
их с готовыми файлами.


In [15]:
expected_text = {}
for sid, row in text_pairs.items():
    decision = decision_by_id.get(sid)
    if decision:
        assert decision["final_status"] == "accepted"
        label = decision["final_label"]
    else:
        assert row["A"]["status"] == row["B"]["status"]
        label = row["A"]["status"]
    expected_text[sid] = (exports["text_A"][sid]["data"]["text"], label)

expected_photos = {}
excluded = set()
for sid, row in image_pairs.items():
    decision = decision_by_id.get(sid)
    if decision and decision["final_status"] == "excluded":
        excluded.add(sid)
        continue
    if decision:
        assert decision["final_status"] == "accepted"
        status, boxes = decision["final_label"], decision["final_boxes"]
    else:
        assert row["A"]["status"] == row["B"]["status"]
        assert len(row["A"]["boxes"]) == len(row["B"]["boxes"])
        status, boxes = row["A"]["status"], row["A"]["boxes"]
    assert status in {"animal", "empty"} and bool(boxes) == (status == "animal")
    size = mapping_by_id[sid]
    width, height = size["width"], size["height"]
    pixel_boxes = [
        [
            box["x"] * width / 100,
            box["y"] * height / 100,
            box["width"] * width / 100,
            box["height"] * height / 100,
        ]
        for box in boxes
    ]
    expected_photos[sid] = {"status": status, "boxes": pixel_boxes}

assert excluded == {"IMG064", "IMG099", "IMG117"}
print("Ожидаем сообщений:", len(expected_text))
print("Ожидаем кадров:", len(expected_photos))
print("Исключены:", ", ".join(sorted(excluded)))


Ожидаем сообщений: 200
Ожидаем кадров: 137
Исключены: IMG064, IMG099, IMG117


In [16]:
final_texts = read_jsonl("data/real/final/texts.jsonl")
coco = read_json("data/real/final/photos/annotations.coco.json")
final_sources = read_jsonl("data/real/final/photos/sources.jsonl")

assert len(final_texts) == len({row["sample_id"] for row in final_texts}) == 200
assert {row["sample_id"] for row in final_texts} == set(expected_text)
for row in final_texts:
    text, label = expected_text[row["sample_id"]]
    assert row["text"] == text and row["label"] == label
    assert row["label_id"] == {"not_spam": 0, "spam": 1}[label]

assert len(coco["images"]) == len({row["sample_id"] for row in coco["images"]}) == 137
assert len({row["id"] for row in coco["images"]}) == 137
assert {row["sample_id"] for row in coco["images"]} == set(expected_photos)
assert coco["categories"] == [{"id": 1, "name": "animal"}]
assert len(coco["annotations"]) == len({box["id"] for box in coco["annotations"]}) == 115
assert {box["image_id"] for box in coco["annotations"]} <= {row["id"] for row in coco["images"]}

for image in coco["images"]:
    sid = image["sample_id"]
    expected = expected_photos[sid]
    assert image["status"] == expected["status"], sid
    assert (image["width"], image["height"]) == (mapping_by_id[sid]["width"], mapping_by_id[sid]["height"])
    boxes = [box for box in coco["annotations"] if box["image_id"] == image["id"]]
    assert len(boxes) == len(expected["boxes"]), sid
    for box, coordinates in zip(boxes, expected["boxes"]):
        assert box["category_id"] == 1 and box["iscrowd"] == 0
        assert all(math.isclose(a, b, abs_tol=1e-5) for a, b in zip(box["bbox"], coordinates)), sid
        x, y, width, height = box["bbox"]
        assert x >= 0 and y >= 0 and width > 0 and height > 0
        assert x + width <= image["width"] + 1e-5 and y + height <= image["height"] + 1e-5
        assert math.isclose(box["area"], width * height)

photo_root = ROOT / "data/real/final/photos"
assert len(final_sources) == len({row["sample_id"] for row in final_sources}) == 137
source_by_id = {row["sample_id"]: row for row in final_sources}
assert set(source_by_id) == set(expected_photos)
for image in coco["images"]:
    sid = image["sample_id"]
    path = photo_root / image["file_name"]
    assert path.is_file(), sid
    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual_hash == source_by_id[sid]["sha256"] == mapping_by_id[sid]["sha256"], sid

listed = {image["file_name"] for image in coco["images"]}
present = {path.relative_to(photo_root).as_posix() for path in (photo_root / "images").iterdir()}
assert listed == present
assert not excluded.intersection(source_by_id)

table(["Итог", "Количество"], [
    ["Сообщения not_spam", sum(row["label"] == "not_spam" for row in final_texts)],
    ["Сообщения spam", sum(row["label"] == "spam" for row in final_texts)],
    ["Кадры animal", sum(row["status"] == "animal" for row in coco["images"])],
    ["Кадры empty", sum(row["status"] == "empty" for row in coco["images"])],
    ["Рамки animal", len(coco["annotations"])],
    ["Исключённые кадры", len(excluded)],
])
print("\nМетки и рамки совпали с решениями. Фотографии совпали с исходными SHA-256.")


Итог               | Количество
-------------------+-----------
Сообщения not_spam | 100       
Сообщения spam     | 100       
Кадры animal       | 71        
Кадры empty        | 66        
Рамки animal       | 115       
Исключённые кадры  | 3         

Метки и рамки совпали с решениями. Фотографии совпали с исходными SHA-256.


Готовые файлы лежат в data/real/final/. Новое пополнение пока хранится отдельно
от исходных train, validation и test. При обучении его можно будет добавлять
в тренировочную часть.

Для повторной проверки достаточно выполнить ячейки этого ноутбука сверху вниз.
Они читают сохранённые данные и выводят результаты здесь же.

Общий журнал сохранён в data/annotation_journal.xlsx. В нём 340 строк,
включая три исключённых кадра. Для каждого объекта собраны источники, ответы
A/B, lead_time и итог сверки. Неизвестные реквизиты источника отмечены в таблице.
